## MLflow Client

In [3]:
from mlflow.tracking import MlflowClient
MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"
client = MlflowClient(tracking_uri = MLFLOW_TRACKING_URI)

In [6]:
client.search_experiments()

[<Experiment: artifact_location='/Users/amruthakaruturi/gitrepos/MLOps/mlruns/1', creation_time=1778078532580, experiment_id='1', last_update_time=1778078532580, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1778077617722, experiment_id='0', last_update_time=1778077617722, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [ ]:
# client.create_experiment(name="experiment_name")

In [28]:
from mlflow.entities import ViewType
runs = client.search_runs(
    experiment_ids = '1',
    filter_string = "",
    run_view_type = ViewType.ACTIVE_ONLY,
    max_results = 4,
    order_by =["attributes.created DESC"]
)

for run in runs:
    print(f"run id: {run.info.run_id}, model: {run.data.tags["model"]}, rmse: {run.data.metrics["rmse"]:.4f}")

run id: e0a57a5861774a22af46af12190a8482, model: LinearSVR, rmse: 782.6816
run id: 1fd1464a976242d192427e7de287ad47, model: GradientBoostingRegressor, rmse: 6.7423
run id: 6c8accbaddf6443492bf015e96ca6dbf, model: RandomForestRegressor, rmse: 6.9050
run id: 68e0565a67554580838cbf3e885d98be, model: XGBoost, rmse: 6.4034


In [27]:
runs[2].data.tags["model"]

'RandomForestRegressor'

## Model registration

In [29]:
import mlflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [44]:
model_name = "nyc-taxi-regressor"

In [ ]:
run_id = "" 
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri = model_uri, name="nyc-taxi-regressor")

In [32]:
# client.list_registered_models()
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1778182815779, deployment_job_id=None, deployment_job_state=None, description='NYC taxi trip duration predictor', last_updated_timestamp=1778184205396, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1778184205396, current_stage='None', deployment_job_state=None, description='', last_updated_timestamp=1778184205396, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='e0a57a5861774a22af46af12190a8482', run_link='', source='models:/m-f0c13768d32447e89f83e5333e57c422', status='READY', status_message=None, tags={'model': 'LinearSVR'}, user_id=None, version=5, workspace='default'>], name='nyc-taxi-regressor', tags={}, workspace='default'>]

In [42]:
latest_versions = client.search_model_versions(f"name='nyc-taxi-regressor'")
for version in latest_versions:
    print(f"version:{version.version}, model_name:{version.tags["model"]}, stage: {version.current_stage}")

version:5, model_name:LinearSVR, stage: None
version:4, model_name:GradientBoostingRegressor, stage: None
version:3, model_name:RandomForestRegressor, stage: None
version:2, model_name:xgboost, stage: None


In [71]:
model_version = 4
new_stage = "Production"
client.transition_model_version_stage(
    name = model_name,
    version = model_version,
    stage = new_stage,
    archive_existing_versions = False
)
from datetime import datetime
date = datetime.today().date()
client.update_model_version(
    name = model_name,
    version = 4,
    description = f"the model version {model_version} was transitioned to {new_stage} on {date}"
)

/var/folders/mh/p31c9s4j7x96gcg6jjhwgmd00000gn/T/ipykernel_64962/2726102277.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1778184188947, current_stage='Production', deployment_job_state=None, description='the model version 4 was transitioned to Production on 2026-05-07', last_updated_timestamp=1778186805311, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='1fd1464a976242d192427e7de287ad47', run_link='', source='models:/m-2bd79b16c2fd4cae85da439674c97e3f', status='READY', status_message=None, tags={'model': 'GradientBoostingRegressor'}, user_id=None, version=4, workspace='default'>

In [59]:
latest_versions = client.search_model_versions(f"name='nyc-taxi-regressor'")
for version in latest_versions:
    print(f"version:{version.version}, model_name:{version.tags["model"]}, stage: {version.current_stage}, run_id: {version.run_id}")

version:4, model_name:GradientBoostingRegressor, stage: Staging, run_id: 1fd1464a976242d192427e7de287ad47
version:5, model_name:LinearSVR, stage: None, run_id: e0a57a5861774a22af46af12190a8482
version:3, model_name:RandomForestRegressor, stage: None, run_id: 6c8accbaddf6443492bf015e96ca6dbf
version:2, model_name:xgboost, stage: None, run_id: 68e0565a67554580838cbf3e885d98be


In [65]:
model_version = 2
new_stage = "Staging"
client.transition_model_version_stage(
    name = model_name,
    version = model_version,
    stage = new_stage,
    archive_existing_versions = False
)
from datetime import datetime
date = datetime.today().date()
client.update_model_version(
    name = model_name,
    version = 2,
    description = f"the model version {model_version} was transitioned to {new_stage} on {date}"
)

/var/folders/mh/p31c9s4j7x96gcg6jjhwgmd00000gn/T/ipykernel_64962/685491861.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1778182892005, current_stage='Staging', deployment_job_state=None, description='the model version 2 was transitioned to Staging on 2026-05-07', last_updated_timestamp=1778186684021, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='68e0565a67554580838cbf3e885d98be', run_link='', source='models:/m-bad72cbb663847479f6d563577531192', status='READY', status_message=None, tags={'model': 'xgboost'}, user_id=None, version=2, workspace='default'>

In [72]:
model_version = 3
new_stage = "Staging"
client.transition_model_version_stage(
    name = model_name,
    version = model_version,
    stage = new_stage,
    archive_existing_versions = False
)
from datetime import datetime
date = datetime.today().date()
client.update_model_version(
    name = model_name,
    version = 3,
    description = f"the model version {model_version} was transitioned to {new_stage} on {date}"
)

/var/folders/mh/p31c9s4j7x96gcg6jjhwgmd00000gn/T/ipykernel_64962/2694540761.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1778184169615, current_stage='Staging', deployment_job_state=None, description='the model version 3 was transitioned to Staging on 2026-05-07', last_updated_timestamp=1778186833436, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='6c8accbaddf6443492bf015e96ca6dbf', run_link='', source='models:/m-fe7da0fed2ee488dbbcb1841e6f725c7', status='READY', status_message=None, tags={'model': 'RandomForestRegressor'}, user_id=None, version=3, workspace='default'>

## Comparing versions and selecting the new "Production" model
Here we will retrieve models registered in the model registry and compare their performance on an unseen test set. The idea is to simulate the scenario in which a deployment engineer has to interact with the model registry to decide whether to update the model version that is in production or not.

These are the steps:

1. Load the test dataset, which corresponds to the NYC Green Taxi data from the month of March 2021.
2. Download the DictVectorizer that was fitted using the training data and saved to MLflow as an artifact, and load it with pickle.
3. Preprocess the test set using the DictVectorizer so we can properly feed the regressors.
4. Make predictions on the test set using the model versions that are currently in the "Staging" and "Production" stages, and compare their performance.
5. Based on the results, update the "Production" model version accordingly.

**Note**: the model registry doesn't actually deploy the model to production when you transition a model to the "Production" stage, it just assign a label to that model version. You should complement the registry with some CI/CD code that does the actual deployment.

In [74]:
import pandas as pd
from sklearn.metrics import root_mean_squared_error

def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)

        df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
        df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    elif filename.endswith('.parquet'):
        df = pd.read_parquet(filename, engine="fastparquet")

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df
    
def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [57]:
df = read_dataframe("data/green_tripdata_2021-03.parquet")

In [60]:
# xgboost
run_id = "68e0565a67554580838cbf3e885d98be"
client.download_artifacts(run_id = run_id, path="preprocessor", dst_path = ".")


'/Users/amruthakaruturi/gitrepos/MLOps/preprocessor'

In [61]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [63]:
X_test = preprocess(df, dv)
target = "duration"
y_test = df[target].values

In [75]:
%time test_model(name=model_name, stage="Production", X_test=X_test, y_test=y_test)

CPU times: user 90.5 ms, sys: 7.87 ms, total: 98.3 ms
Wall time: 104 ms


{'rmse': 6.659623830022514}

In [76]:
%time test_model(name=model_name, stage="Staging", X_test=X_test, y_test=y_test)

CPU times: user 5.65 s, sys: 395 ms, total: 6.04 s
Wall time: 6.34 s


{'rmse': 6.878501821845514}

In [77]:
import mlflow

# 1. Pull EVERY version for this model
versions = client.search_model_versions(f"name='{model_name}'")

# 2. Iterate and filter for those in "Staging"
for v in versions:
    if v.current_stage == "Staging":
        print(f"\n--- Testing Model Version: {v.version} ---")
        
        # CRITICAL: Load by Version Number, not by Stage string
        model_uri = f"models:/{model_name}/{v.version}"
        model = mlflow.pyfunc.load_model(model_uri)
        
        # 3. Run your evaluation
        y_pred = model.predict(X_test)
        rmse = root_mean_squared_error(y_test, y_pred)
        
        print(f"Version {v.version} RMSE: {rmse}")
        
        # Optional: Log the results to a "Champion-Challenger" run
        # with mlflow.start_run(run_name=f"test_v{v.version}"):
        #     mlflow.log_metric("test_rmse", rmse)


--- Testing Model Version: 3 ---
Version 3 RMSE: 6.878501821845514

--- Testing Model Version: 2 ---
Version 2 RMSE: 6.349443553315019


In [78]:
client.transition_model_version_stage(
    name=model_name,
    version=2,
    stage="Production",
    archive_existing_versions=True
)

/var/folders/mh/p31c9s4j7x96gcg6jjhwgmd00000gn/T/ipykernel_64962/1703833061.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1778182892005, current_stage='Production', deployment_job_state=None, description='the model version 2 was transitioned to Staging on 2026-05-07', last_updated_timestamp=1778187043592, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='68e0565a67554580838cbf3e885d98be', run_link='', source='models:/m-bad72cbb663847479f6d563577531192', status='READY', status_message=None, tags={'model': 'xgboost'}, user_id=None, version=2, workspace='default'>